# Prepare Clinical Data for Competing Risks: TCGA PanCancer Clinical Data

This repository contains the combined clinical data with follow up-and outcome
information for the TCGA PanCancer Atlas in a sinlge text file and RData file.
All data in its original format can be found at
<https://gdc.cancer.gov/about-data/publications/pancanatlas>. 

### The PanCancer Atlas

The original
[paper](https://www.sciencedirect.com/science/article/pii/S0092867418303027?via%3Dihub) (Cell-of-Origin Patterns Dominate the Molecular Classification of
10,000 Tumors from 33 Types of Cancer by Hoadley et al) represented
efforts to provide “comprehensive integrative molecular analyses of the
complete set of tumors in TCGA”. This paper was accompanied by a paper
from [Liu et
al](https://www.cell.com/cell/pdf/S0092-8674\(18\)30229-0.pdf) (An
Integrated TCGA Pan-Cancer Clinical Data Resource to Drive High-Quality
Survival Outcome Analytics) who attempted to create a standardized
dataset for the clinical data across the PanCancer Atlas called the TCGA
Pan-Cancer Clinical Data Resource (TCGA-CDR). It provides two data resources for clinical
annotations: `clinical_PANCAN_patient_with_followup.tsv` (which is
available under Additional Resources/Supplemental Data and is not
assocaited with a specific publication and details on it’s creation are
scarce) and `TCGA-CDR-SupplementalTableS1.xlsx` which is described as a
“curated resource of the clinical annotations for TCGA data and
provides recommendations for use of clinical endpoints” and comed with
the recommendation that “this file be used for clinical elements and
survival outcome data first” and is associated with the Liu et al
publication. 

Survival Endpoints:

  - OS - overall survival
  - DSS - disease-specific survival  
  - DFI - disease-free interval
  - PFI - progression-free
interval

For clinical outcome endpoints, they recommend the use of **PFI** for
progression-free interval, and **OS** for overall survival. Both
endpoints are relatively accurate. Given the relatively short follow-up
time, PFI is preferred over OS. For that reason, we use **PFI**. For the detailed definitions and interpretation of competing risks events, please refer to the original article and the supplemental notes. These resources recommend using the PFI endpoint for analysis and provide precise explanations of the event coding: 0 indicates censored observations, 1 denotes the event of interest (such as tumor progression or cancer-related death), and 2 represents competing risk events (death without prior progression). This standardized coding ensures accurate modeling of progression outcomes in the presence of competing risks.

In [1]:
import pandas as pd
from tabulate import tabulate
import os

In [2]:
# File path
file_path = "../data_download/clinical_PANCAN/TCGA-CDR-SupplementalTableS1.xlsx"

# Load the Excel file with multiple sheets
xls = pd.ExcelFile(file_path)

# Display all available sheet names
print("Available sheets:", xls.sheet_names)

# Load the third sheet (index 2, since indexing starts at 0)
df_extra = pd.read_excel(xls, sheet_name=xls.sheet_names[2])

# Display the first few rows
print(df_extra.head())

# Save the sheet to a CSV file in the same directory as the Excel file
output_path = "../data_download/clinical_PANCAN/ExtraEndpoints.csv"
df_extra.to_csv(output_path, index=False)


Available sheets: ['TCGA-CDR', 'TCGA-CDR_Notes', 'ExtraEndpoints', 'ExtraEndpoints_Notes', 'Table4_PHAssumptionTests', 'Table5_PHAssumptionTests', 'TSS_Info', 'Fig2EFG_AdditionalInfo']
   Unnamed: 0 bcr_patient_barcode type  PFI.1  PFI.time.1  PFI.2  PFI.time.2  \
0           1        TCGA-OR-A5J1  ACC    1.0       754.0    1.0       754.0   
1           2        TCGA-OR-A5J2  ACC    1.0       289.0    1.0       289.0   
2           3        TCGA-OR-A5J3  ACC    1.0        53.0    1.0        53.0   
3           4        TCGA-OR-A5J4  ACC    1.0       126.0    1.0       126.0   
4           5        TCGA-OR-A5J5  ACC    1.0        50.0    1.0        50.0   

   PFS  PFS.time  DSS_cr  DSS.time.cr  DFI.cr  DFI.time.cr  PFI.cr  \
0  1.0     754.0     1.0       1355.0     1.0        754.0     1.0   
1  1.0     289.0     1.0       1677.0     NaN          NaN     1.0   
2  1.0      53.0     0.0       2091.0     1.0         53.0     1.0   
3  1.0     126.0     1.0        423.0     NaN         

## Filtering Clinical Data for BRCA and LGG


In [3]:
# File path
file_path = "../data_download/clinical_PANCAN/ExtraEndpoints.csv"

# Load the file
df = pd.read_csv(file_path)

# Filter for BRCA and LGG
df_brca = df[df['type'] == 'BRCA'].copy()
df_lgg = df[df['type'] == 'LGG'].copy()

# Define output directories
output_base = "../data_download/clinical_PANCAN"
brca_path = os.path.join(output_base, "BRCA")
lgg_path = os.path.join(output_base, "LGG")

# Create directories if they don't exist
os.makedirs(brca_path, exist_ok=True)
os.makedirs(lgg_path, exist_ok=True)

# Save filtered data to CSV
brca_file = os.path.join(brca_path, "cr_BRCA.csv")
lgg_file = os.path.join(lgg_path, "cr_LGG.csv")

df_brca.to_csv(brca_file, index=False)
df_lgg.to_csv(lgg_file, index=False)

# Display LGG data
display(df_lgg)

# Print confirmation messages
print(f"BRCA: {df_brca.shape[0]} rows saved to {brca_file}")
print(f"LGG: {df_lgg.shape[0]} rows saved to {lgg_file}")


,Unnamed: 0,bcr_patient_barcode,type,PFI.1,PFI.time.1,PFI.2,PFI.time.2,PFS,PFS.time,DSS_cr,DSS.time.cr,DFI.cr,DFI.time.cr,PFI.cr,PFI.time.cr,PFI.1.cr,PFI.time.1.cr,PFI.2.cr,PFI.time.2.cr
4910,4911,TCGA-CS-4938,LGG,0.0,3574.0,0.0,3574.0,0.0,3574.0,0.0,3574.0,NaN,NaN,0.0,3574.0,0.0,3574.0,0.0,3574.0
4911,4912,TCGA-CS-4941,LGG,1.0,9.0,NaN,NaN,1.0,9.0,1.0,234.0,NaN,NaN,1.0,9.0,1.0,9.0,NaN,NaN
4912,4913,TCGA-CS-4942,LGG,1.0,1184.0,NaN,NaN,1.0,1184.0,1.0,1335.0,NaN,NaN,1.0,1184.0,1.0,1184.0,NaN,NaN
4913,4914,TCGA-CS-4943,LGG,1.0,1106.0,1.0,1106.0,1.0,1106.0,1.0,1106.0,NaN,NaN,1.0,1106.0,1.0,1106.0,1.0,1106.0
4914,4915,TCGA-CS-4944,LGG,0.0,1828.0,0.0,1828.0,0.0,1828.0,0.0,1828.0,NaN,NaN,0.0,1828.0,0.0,1828.0,0.0,1828.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5420,5421,TCGA-WY-A85A,LGG,0.0,1320.0,0.0,1320.0,0.0,1320.0,0.0,1320.0,NaN,NaN,0.0,1320.0,0.0,1320.0,0.0,1320.0
5421,5422,TCGA-WY-A85B,LGG,0.0,1393.0,0.0,1393.0,0.0,1393.0,0.0,1393.0,0.0,1393.0,0.0,1393.0,0.0,1393.0,0.0,1393.0
5422,5423,TCGA-WY-A85C,LGG,1.0,809.0,NaN,NaN,1.0,809.0,0.0,1426.0,NaN,NaN,1.0,809.0,1.0,809.0,NaN,NaN
5423,5424,TCGA-WY-A85D,LGG,1.0,887.0,NaN,NaN,1.0,887.0,0.0,1147.0,NaN,NaN,1.0,887.0,1.0,887.0,NaN,NaN


BRCA: 1097 rows saved to ../data_download/clinical_PANCAN/BRCA/cr_BRCA.csv
LGG: 515 rows saved to ../data_download/clinical_PANCAN/LGG/cr_LGG.csv


In [4]:
# Define cancer types and their paths
cancer_types = ["BRCA", "LGG"] 
base_path = "../data_download/clinical_PANCAN"

# Process each cancer type
for cancer in cancer_types:
    # Define file paths
    tsv_path = os.path.join(base_path, f"filtered{cancer}_sample_sheet.processed_with_cr.tsv")
    csv_path = os.path.join(base_path, cancer, f"cr_{cancer}.csv")

    # Load data
    tsv_df = pd.read_csv(tsv_path, sep="\t")
    csv_df = pd.read_csv(csv_path)

    # Merge on patient identifier
    merged_df = tsv_df.merge(
        csv_df[['bcr_patient_barcode', 'PFI.cr', 'PFI.time.cr']],
        left_on='case_submitter_id',
        right_on='bcr_patient_barcode',
        how='left'
    )

    # Drop redundant identifier column
    merged_df.drop(columns=['bcr_patient_barcode'], inplace=True)

    # Save the merged file with added columns
    output_path = os.path.join(base_path, f"filtered{cancer}_sample_sheet.processed_with_cr.tsv")
    merged_df.to_csv(output_path, sep="\t", index=False)
 


/tmp/ipykernel_1384287/1082877653.py:16: FutureWarning: Passing 'suffixes' which cause duplicate columns {'PFI.cr_x', 'PFI.time.cr_x'} in the result is deprecated and will raise a MergeError in a future version.
  merged_df = tsv_df.merge(
/tmp/ipykernel_1384287/1082877653.py:16: FutureWarning: Passing 'suffixes' which cause duplicate columns {'PFI.cr_x', 'PFI.time.cr_x'} in the result is deprecated and will raise a MergeError in a future version.
  merged_df = tsv_df.merge(


In [5]:
# Store the current display setting for max column width
old_max_colwidth = pd.get_option('display.max_colwidth')

# Load only the sheet named "ExtraEndpoints_Notes"
df = pd.read_excel("../data_download/clinical_PANCAN/TCGA-CDR-SupplementalTableS1.xlsx", sheet_name="ExtraEndpoints_Notes", header=None)

# Filter rows that start with 'PFI.1.cr' or 'PFI.time.1.cr'
mask = df[0].astype(str).str.startswith(("PFI.1.cr", "PFI.time.1.cr"))
filtered_rows = df[mask][0]

# Display the filtered results
print(filtered_rows)

# Restore the original display setting
pd.set_option('display.max_colwidth', old_max_colwidth)


18    PFI.1.cr: progression-free interval (PFI.1) wi...
19    PFI.time.1.cr: progression-free interval time ...
Name: 0, dtype: object


In [6]:
# Base output path for saving clinical data
output_base_path = "../samvae-main/data_preprocessing/raw_data/clinical_data"

# Process each cancer type
for cancer in cancer_types:
    cancer_lower = cancer.lower()
    output_dir = os.path.join(output_base_path, cancer_lower)
    os.makedirs(output_dir, exist_ok=True)

    # Load the merged file created previously
    merged_file = os.path.join(base_path, f"filtered{cancer}_sample_sheet.processed_with_cr.tsv")
    df_merged = pd.read_csv(merged_file, sep="\t")

    # Define output path
    output_file = os.path.join(output_dir, f"{cancer_lower}_clinical_cr.csv")

    # Save the merged file as a CSV
    df_merged.to_csv(output_file, index=False)

## Other Files

In [7]:
# Original file path
file_path = "../data_download/clinical_PANCAN/clinical_PANCAN_patient_with_followup.tsv"

# Load the TSV file into a DataFrame with an alternative encoding
df = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1', low_memory=False)

# Filter the data by the 'acronym' column
df_brca = df[df['acronym'] == 'BRCA']
df_lgg = df[df['acronym'] == 'LGG']

# Define the output save paths
output_path_brca = "../data_download/clinical_PANCAN/clinical_PANCAN_BRCA.tsv"
output_path_lgg = "../data_download/clinical_PANCAN/clinical_PANCAN_LGG.tsv"

# Save as TSV format
df_brca.to_csv(output_path_brca, sep='\t', index=False)
df_lgg.to_csv(output_path_lgg, sep='\t', index=False)


In [8]:
def process_clinical_data(cancer_type):
    base_path = "/home/alba/snap/snapd-desktop-integration/tfm/data_download"
    manifest_path = f"{base_path}/Manifest/{cancer_type}/filtered_manifest/filtered_gdc_manifest.clinical.txt"
    sample_sheet_path = f"{base_path}/clinical_PANCAN/{cancer_type}/clinical_PANCAN.tsv"
    
    # Load the files
    manifest_df = pd.read_csv(manifest_path, sep="\t")
    sample_sheet_df = pd.read_csv(sample_sheet_path, sep="\t")
    
    # Filter rows where the "bcr_patient_barcode" in the sample sheet is in the manifest
    filtered_df = sample_sheet_df[sample_sheet_df['bcr_patient_barcode'].isin(manifest_df['case_submitter_id'])].copy()
    
    # Rows that are in the sample sheet but not in the manifest
    not_in_manifest_df = sample_sheet_df[~sample_sheet_df['bcr_patient_barcode'].isin(manifest_df['case_submitter_id'])].copy()
    
    print(f"=== {cancer_type} ===")
    print(f"Total in sample sheet: {len(sample_sheet_df)}")
    print(f"Rows found in manifest: {len(filtered_df)}") 
    
    return filtered_df, not_in_manifest_df

# Process LGG and BRCA and store the resulting DataFrames
df_lgg, not_in_manifest_lgg = process_clinical_data("LGG")
df_brca, not_in_manifest_brca = process_clinical_data("BRCA")


=== LGG ===
Total in sample sheet: 515
Rows found in manifest: 260
=== BRCA ===
Total in sample sheet: 1099
Rows found in manifest: 529


/tmp/ipykernel_1384287/3595895105.py:8: DtypeWarning: Columns (27,176,177,257,472,580) have mixed types. Specify dtype option on import or set low_memory=False.
  sample_sheet_df = pd.read_csv(sample_sheet_path, sep="\t")


# LGG

In [9]:
print("LGG Clinical Data:")
display(df_lgg)  


LGG Clinical Data:


,bcr_patient_uuid,bcr_patient_barcode,acronym,gender,vital_status,days_to_birth,days_to_death,days_to_last_followup,days_to_initial_pathologic_diagnosis,age_at_initial_pathologic_diagnosis,...,total_bilirubin_upper_limit,platelet_result_count,fibrosis_ishak_score,fetoprotein_outcome_value,fetoprotein_outcome_upper_limit,fetoprotein_outcome_lower_limit,inter_norm_ratio_lower_limit,family_cancer_type_txt,bilirubin_upper_limit,days_to_last_known_alive
0,334f715e-08dc-4a29-b8e4-b010b829c478,TCGA-CS-4938,LGG,FEMALE,Alive,-11509,[Not Applicable],3574.0,0,31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,230f5fa7-aa36-41ea-b40b-08f520767bd5,TCGA-CS-4942,LGG,FEMALE,Dead,-16297,1335.0,[Not Available],0,44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,64cd17eb-c778-45e9-b994-02b68182e51b,TCGA-CS-4944,LGG,MALE,Alive,-18494,[Not Applicable],1828.0,0,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,3f70c3e3-0131-466f-92aa-0a63ab3d4258,TCGA-CS-6188,LGG,MALE,Dead,-17729,814.0,725.0,0,48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,42fe260f-520b-4fac-8c09-d2b31aed59fb,TCGA-CS-6290,LGG,MALE,Dead,-11666,1137.0,546.0,0,31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506,732DF38E-D32C-4D63-AC97-AD655E2E5732,TCGA-W9-A837,LGG,MALE,Alive,[Not Available],[Not Applicable],1553.0,0,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
507,42156C6A-416E-4832-9C95-CEE6084B0910,TCGA-WH-A86K,LGG,MALE,Alive,-24055,[Not Applicable],405.0,0,65,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510,09552B53-4393-4811-A671-A9EF6EFF4790,TCGA-WY-A85A,LGG,MALE,Alive,-7380,[Not Applicable],1320,0,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
513,0D5A4C63-C3FB-4295-A433-D90CDBFC4ED6,TCGA-WY-A85D,LGG,MALE,Alive,-21979,[Not Applicable],1147,0,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# BRCA

In [10]:
print("\nBRCA Clinical Data:")
display(df_brca)


BRCA Clinical Data:


,bcr_patient_uuid,bcr_patient_barcode,acronym,gender,vital_status,days_to_birth,days_to_death,days_to_last_followup,days_to_initial_pathologic_diagnosis,age_at_initial_pathologic_diagnosis,...,total_bilirubin_upper_limit,platelet_result_count,fibrosis_ishak_score,fetoprotein_outcome_value,fetoprotein_outcome_upper_limit,fetoprotein_outcome_lower_limit,inter_norm_ratio_lower_limit,family_cancer_type_txt,bilirubin_upper_limit,days_to_last_known_alive
0,6E7D5EC6-A469-467C-B748-237353C23416,TCGA-3C-AAAU,BRCA,FEMALE,Alive,-20211,[Not Applicable],4047.0,0,55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,55262FCB-1B01-4480-B322-36570430C917,TCGA-3C-AALI,BRCA,FEMALE,Alive,-18538,[Not Applicable],4005.0,0,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,427D0648-3F77-4FFC-B52C-89855426D647,TCGA-3C-AALJ,BRCA,FEMALE,Alive,-22848,[Not Applicable],1474.0,0,62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6623FC5E-00BE-4476-967A-CBD55F676EA6,TCGA-4H-AAAK,BRCA,FEMALE,Alive,-18371,[Not Applicable],348.0,0,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,16FC3677-0393-4ED1-AD3F-C8355F056369,TCGA-5L-AAT1,BRCA,FEMALE,Alive,-23225,[Not Applicable],1471,0,63,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1089,4B54E06E-A280-4981-A4E1-9AEA154341B4,TCGA-UL-AAZ6,BRCA,FEMALE,Alive,-26999,[Not Applicable],518.0,0,73,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1090,45013972-2DFD-4F82-A076-E3E4AF1B43B8,TCGA-UU-A93S,BRCA,FEMALE,Dead,-23278,116,[Not Available],0,63,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1091,1285EB55-415C-494A-AA58-936F0427CDD0,TCGA-V7-A7HQ,BRCA,FEMALE,Alive,-27684,[Not Applicable],2033.0,0,75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1094,5CD79093-1571-4F71-8136-0D84CCABDCAC,TCGA-WT-AB44,BRCA,FEMALE,Alive,[Not Available],[Not Applicable],883.0,0,77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
